# MCP-IDS: Graph-Based Attack Detection for LLM Agent Sessions

This notebook runs all experiments for the MCP-IDS paper:
- **3 datasets**: RAS-Eval, ATBench, Combined (RAS-Eval + ATBench + mcpbench)
- **4 architectures**: GAT, GCN, GraphSAGE, MLP (no-graph baseline)
- **2 split protocols**: Task-stratified (RAS-Eval), Label-stratified (all)
- **Metrics**: AUROC, AUPRC, Macro F1, Precision, Recall, FPR
- **Breakdowns**: Per-attack-type, per-source, per-agent

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q torch torch-geometric sentence-transformers scikit-learn numpy matplotlib

In [ ]:
import os
import json
import hashlib
import numpy as np
import torch
import torch.nn.functional as F
from collections import Counter
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score,
)
from sklearn.model_selection import train_test_split
from torch_geometric.data import Data
from torch_geometric.nn import GATConv, GCNConv, SAGEConv, global_mean_pool, global_max_pool
from torch_geometric.loader import DataLoader

SEEDS = [7, 42, 123]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 2. Download Datasets

In [ ]:
# Mount Google Drive (datasets stored there)
from google.colab import drive
drive.mount('/content/drive')

# Set paths - adjust if your Drive structure differs
PROJECT_DIR = '/content/drive/MyDrive/mcp-agent-attack-detection'
DATA_DIR = os.path.join(PROJECT_DIR, 'data/raw')

# Alternatively, download datasets directly:
# !git clone https://github.com/... data/raw/ras_eval/_repo
# !wget ... -O data/raw/atbench/atbench.json

print(f"Data dir: {DATA_DIR}")
print(f"Contents: {os.listdir(DATA_DIR)}")

## 3. Data Loading

In [ ]:
# ---- RAS-Eval Loader ----

def _extract_tool_calls_raseval(record):
    calls = []
    pending = []
    for msg in record.get("response", []):
        if not isinstance(msg, dict):
            continue
        if msg.get("type") == "AIMessage" and "tool_calls" in msg:
            for tc in msg["tool_calls"]:
                pending.append({
                    "name": tc.get("name", "unknown"),
                    "args_str": json.dumps(tc.get("args", {})),
                })
        elif msg.get("type") == "ToolMessage" and pending:
            ci = pending.pop(0)
            content = str(msg.get("content", ""))
            calls.append({
                "tool_name": ci["name"],
                "args_str": ci["args_str"],
                "response_str": content,
                "response_length": len(content),
                "params_hash": int(hashlib.md5(ci["args_str"].encode()).hexdigest(), 16) % 10000,
            })
    return calls


def load_ras_eval(raw_dir, pooled=True):
    attack_tasks_path = os.path.join(raw_dir, "_repo", "data", "tasks", "attack_tasks.json")
    with open(attack_tasks_path) as f:
        attack_tasks = json.load(f)
    attack_mode_map = {}
    for t in attack_tasks:
        modes = sorted(set(a["mode"] for a in t["attack"]))
        attack_mode_map[t["index"]] = "+".join(modes)

    tasks_path = os.path.join(raw_dir, "_repo", "data", "tasks", "tasks.json")
    with open(tasks_path) as f:
        tasks = json.load(f)
    agent_map = {t["index"]: t["agent"] for t in tasks}
    target_map = {t["index"]: t["target_index"] for t in attack_tasks}

    benign = []
    logs_dir = os.path.join(raw_dir, "logs")
    exclude = {"guard_response.jsonl"}
    for fname in sorted(os.listdir(logs_dir)):
        if not fname.endswith(".jsonl") or fname in exclude:
            continue
        if not pooled and fname != "glm-4-flash.jsonl":
            continue
        with open(os.path.join(logs_dir, fname)) as f:
            for line in f:
                if not line.strip():
                    continue
                r = json.loads(line)
                calls = _extract_tool_calls_raseval(r)
                if not calls:
                    continue
                task_id = r.get("index", r.get("id", -1))
                benign.append({
                    "calls": calls, "label": 0,
                    "task_id": f"ras_{task_id}", "source": "ras_eval",
                    "attack_type": "benign",
                    "agent": agent_map.get(task_id, "unknown"),
                })

    attacked = []
    attacked_path = os.path.join(raw_dir, "attacked", "glm-4-flash.jsonl")
    with open(attacked_path) as f:
        for line in f:
            if not line.strip():
                continue
            r = json.loads(line)
            calls = _extract_tool_calls_raseval(r)
            if not calls:
                continue
            atk_idx = r["index"]
            target = target_map.get(atk_idx, -1)
            attacked.append({
                "calls": calls, "label": 1,
                "task_id": f"ras_{target}", "source": "ras_eval",
                "attack_type": attack_mode_map.get(atk_idx, "unknown"),
                "agent": agent_map.get(target, "unknown"),
            })

    print(f"[RAS-Eval] {len(benign)} benign + {len(attacked)} attacked")
    return benign, attacked

In [ ]:
# ---- ATBench Loader ----

def _extract_tool_calls_atbench(entry):
    calls = []
    messages = entry["contents"][0]
    actions, env_responses = [], []
    for msg in messages:
        if msg.get("role") == "agent" and msg.get("action"):
            action = msg["action"]
            if action.startswith("Complete"):
                continue
            try:
                actions.append(json.loads(action))
            except json.JSONDecodeError:
                continue
        elif msg.get("role") == "environment":
            env_responses.append(str(msg.get("content", "")))
    for i, action in enumerate(actions):
        args_str = json.dumps(action.get("arguments", {}))
        response_str = env_responses[i] if i < len(env_responses) else ""
        calls.append({
            "tool_name": action.get("name", "unknown"),
            "args_str": args_str,
            "response_str": response_str,
            "response_length": len(response_str),
            "params_hash": int(hashlib.md5(args_str.encode()).hexdigest(), 16) % 10000,
        })
    return calls


def load_atbench(raw_dir):
    path = os.path.join(raw_dir, "atbench.json")
    with open(path) as f:
        data = json.load(f)
    benign, attacked = [], []
    for entry in data:
        calls = _extract_tool_calls_atbench(entry)
        if not calls:
            continue
        session = {
            "calls": calls, "label": entry["label"],
            "task_id": f"atb_{entry['id']}", "source": "atbench",
            "attack_type": entry.get("risk_source", "benign") if entry["label"] == 1 else "benign",
            "agent": entry.get("failure_mode", "unknown") if entry["label"] == 1 else "safe",
        }
        if entry["label"] == 0:
            benign.append(session)
        else:
            attacked.append(session)
    print(f"[ATBench] {len(benign)} benign + {len(attacked)} attacked")
    return benign, attacked

In [ ]:
# ---- mcpbench Loader ----

def _extract_tool_calls_mcpbench(record):
    calls = []
    messages = record.get("messages", [])
    pending = []
    for msg in messages:
        if msg.get("role") == "assistant" and "tool_calls" in msg:
            for tc in msg["tool_calls"]:
                func = tc.get("function", {})
                pending.append({
                    "id": tc.get("id", ""),
                    "name": func.get("name", "unknown"),
                    "args_str": func.get("arguments", "{}"),
                })
        elif msg.get("role") == "tool" and pending:
            tc_id = msg.get("tool_call_id", "")
            matched = None
            for i, p in enumerate(pending):
                if p["id"] == tc_id:
                    matched = pending.pop(i)
                    break
            if matched is None and pending:
                matched = pending.pop(0)
            if matched:
                content = str(msg.get("content", ""))
                calls.append({
                    "tool_name": matched["name"],
                    "args_str": matched["args_str"],
                    "response_str": content,
                    "response_length": len(content),
                    "params_hash": int(hashlib.md5(matched["args_str"].encode()).hexdigest(), 16) % 10000,
                })
    return calls


def load_mcpbench(raw_dir):
    path = os.path.join(raw_dir, "mcpbench.jsonl")
    sessions = []
    with open(path) as f:
        for line in f:
            if not line.strip():
                continue
            r = json.loads(line)
            calls = _extract_tool_calls_mcpbench(r)
            if not calls:
                continue
            sessions.append({
                "calls": calls, "label": 0,
                "task_id": f"mcp_{r.get('task_id', r.get('id', 'unk'))}",
                "source": "mcpbench", "attack_type": "benign",
                "agent": r.get("domain", "unknown"),
            })
    print(f"[mcpbench] {len(sessions)} benign sessions")
    return sessions

In [ ]:
# Load all datasets
ras_b, ras_a = load_ras_eval(os.path.join(DATA_DIR, 'ras_eval'), pooled=True)
atb_b, atb_a = load_atbench(os.path.join(DATA_DIR, 'atbench'))
mcp_b = load_mcpbench(os.path.join(DATA_DIR, 'cx_cmu'))

print(f"\nTotal: {len(ras_b)+len(atb_b)+len(mcp_b)} benign + {len(ras_a)+len(atb_a)} attacked")

## 4. Graph Construction

In [ ]:
# ---- Feature Extraction ----

def build_tool_vocab(sessions):
    tools = set()
    for s in sessions:
        for c in s["calls"]:
            tools.add(c["tool_name"])
    return {t: i for i, t in enumerate(sorted(tools))}


def metadata_features(call, tool_vocab, n_tools):
    onehot = np.zeros(n_tools, dtype=np.float32)
    idx = tool_vocab.get(call["tool_name"], -1)
    if idx >= 0:
        onehot[idx] = 1.0
    return np.concatenate([onehot, [call["params_hash"] / 10000.0],
                           [min(call["response_length"], 10000) / 10000.0]])


def get_content_embedder():
    from sentence_transformers import SentenceTransformer
    return SentenceTransformer("all-MiniLM-L6-v2", device=str(device))


def content_features(call, embedder):
    args_emb = embedder.encode(call["args_str"][:512], show_progress_bar=False)
    resp_text = call["response_str"][:512] if call["response_str"] else "empty"
    resp_emb = embedder.encode(resp_text, show_progress_bar=False)
    return np.concatenate([args_emb, resp_emb]).astype(np.float32)


def build_edges(calls):
    n = len(calls)
    src, dst = [], []
    # Sequential edges (bidirectional)
    for i in range(n - 1):
        src.extend([i, i + 1])
        dst.extend([i + 1, i])
    # Data-flow edges
    for i in range(n):
        resp = calls[i]["response_str"]
        if not resp or len(resp) > 1000:
            continue
        for j in range(i + 1, n):
            args = calls[j]["args_str"]
            if resp[:50] in args or any(
                v in args for v in resp.split()[:5] if len(v) > 4
            ):
                src.extend([i, j])
                dst.extend([j, i])
    if not src:
        src, dst = [0], [0]
    return torch.tensor([src, dst], dtype=torch.long)


def sessions_to_graphs(sessions, mode, tool_vocab, n_tools, embedder=None):
    graphs = []
    for idx, s in enumerate(sessions):
        calls = s["calls"]
        node_feats = []
        for c in calls:
            parts = []
            if mode in ("metadata", "both"):
                parts.append(metadata_features(c, tool_vocab, n_tools))
            if mode in ("content", "both"):
                parts.append(content_features(c, embedder))
            node_feats.append(np.concatenate(parts))
        x = torch.tensor(np.stack(node_feats), dtype=torch.float)
        edge_index = build_edges(calls)
        y = torch.tensor([s["label"]], dtype=torch.long)
        data = Data(x=x, edge_index=edge_index, y=y)
        data.task_id = s["task_id"]
        graphs.append(data)
        if (idx + 1) % 500 == 0:
            print(f"  Processed {idx + 1}/{len(sessions)} sessions")
    print(f"  Built {len(graphs)} graphs, feature dim = {graphs[0].x.shape[1]}")
    return graphs

## 5. GNN Models

In [ ]:
def build_conv(arch, in_dim, out_dim, heads=4):
    if arch == "gat":
        return GATConv(in_dim, out_dim, heads=heads, concat=False)
    elif arch == "gcn":
        return GCNConv(in_dim, out_dim)
    elif arch == "sage":
        return SAGEConv(in_dim, out_dim)
    else:
        raise ValueError(f"Unknown arch: {arch}")


class GNNClassifier(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, arch="gat", heads=4):
        super().__init__()
        self.conv1 = build_conv(arch, in_dim, hidden_dim, heads)
        self.conv2 = build_conv(arch, hidden_dim, hidden_dim, heads)
        self.lin1 = torch.nn.Linear(hidden_dim * 2, hidden_dim)
        self.lin2 = torch.nn.Linear(hidden_dim, 2)
        self.dropout = torch.nn.Dropout(0.3)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.elu(self.conv1(x, edge_index))
        x = self.dropout(x)
        x = F.elu(self.conv2(x, edge_index))
        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)
        x = torch.cat([x_mean, x_max], dim=1)
        x = F.elu(self.lin1(x))
        x = self.dropout(x)
        x = self.lin2(x)
        return x


class MLPClassifier(torch.nn.Module):
    """No-graph baseline: mean+max pool raw node features, then MLP."""
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.lin1 = torch.nn.Linear(in_dim * 2, hidden_dim)
        self.lin2 = torch.nn.Linear(hidden_dim, hidden_dim)
        self.lin3 = torch.nn.Linear(hidden_dim, 2)
        self.dropout = torch.nn.Dropout(0.3)

    def forward(self, data):
        x, batch = data.x, data.batch
        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)
        x = torch.cat([x_mean, x_max], dim=1)
        x = F.elu(self.lin1(x))
        x = self.dropout(x)
        x = F.elu(self.lin2(x))
        x = self.dropout(x)
        x = self.lin3(x)
        return x

## 6. Training & Evaluation Functions

In [ ]:
def train_epoch(model, loader, optimizer, device, class_weights=None):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        loss = F.cross_entropy(out, batch.y.view(-1), weight=class_weights)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    for batch in loader:
        batch = batch.to(device)
        out = model(batch)
        probs = F.softmax(out, dim=1)[:, 1]
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(batch.y.view(-1).cpu().numpy())
    return np.array(all_probs), np.array(all_labels)


def compute_metrics(probs, labels):
    results = {}
    if len(np.unique(labels)) < 2:
        return results
    preds = (probs > 0.5).astype(int)
    results["auroc"] = float(roc_auc_score(labels, probs))
    results["auprc"] = float(average_precision_score(labels, probs))
    results["macro_f1"] = float(f1_score(labels, preds, average="macro"))
    results["weighted_f1"] = float(f1_score(labels, preds, average="weighted"))
    results["precision"] = float(precision_score(labels, preds, zero_division=0))
    results["recall"] = float(recall_score(labels, preds))
    tp = ((preds == 1) & (labels == 1)).sum()
    fp = ((preds == 1) & (labels == 0)).sum()
    tn = ((preds == 0) & (labels == 0)).sum()
    results["fpr"] = float(fp / max(fp + tn, 1))
    results["accuracy"] = float((preds == labels).mean())
    return results


def per_group_metrics(probs, labels, groups):
    preds = (probs > 0.5).astype(int)
    breakdown = {}
    for gid in sorted(set(groups)):
        mask = np.array(groups) == gid
        if mask.sum() == 0:
            continue
        g_labels = labels[mask]
        g_preds = preds[mask]
        g_probs = probs[mask]
        n_attack = int((g_labels == 1).sum())
        n_benign = int((g_labels == 0).sum())
        stats = {"n_total": int(mask.sum()), "n_attack": n_attack, "n_benign": n_benign}
        if n_attack > 0:
            stats["recall"] = float(((g_preds == 1) & (g_labels == 1)).sum() / n_attack)
        if n_benign > 0:
            stats["fpr"] = float(((g_preds == 1) & (g_labels == 0)).sum() / n_benign)
        if len(np.unique(g_labels)) > 1:
            stats["auroc"] = float(roc_auc_score(g_labels, g_probs))
        breakdown[gid] = stats
    return breakdown

In [ ]:
# ---- Split Functions ----

def label_stratified_split(n, labels, seed=42):
    """70/10/20 stratified by label."""
    idx = np.arange(n)
    train_idx, temp_idx = train_test_split(idx, test_size=0.3, stratify=labels, random_state=seed)
    temp_labels = labels[temp_idx]
    val_idx, test_idx = train_test_split(temp_idx, test_size=2/3, stratify=temp_labels, random_state=seed)
    return train_idx, val_idx, test_idx


def task_stratified_split(n, task_ids, seed=42):
    """70/10/20 with tasks disjoint across splits."""
    unique_tasks = sorted(set(task_ids))
    rng = np.random.RandomState(seed)
    rng.shuffle(unique_tasks)
    nt = len(unique_tasks)
    n_train = int(0.7 * nt)
    n_val = max(1, int(0.1 * nt))
    train_tasks = set(unique_tasks[:n_train])
    val_tasks = set(unique_tasks[n_train:n_train + n_val])
    test_tasks = set(unique_tasks[n_train + n_val:])
    train_idx = np.array([i for i in range(n) if task_ids[i] in train_tasks])
    val_idx = np.array([i for i in range(n) if task_ids[i] in val_tasks])
    test_idx = np.array([i for i in range(n) if task_ids[i] in test_tasks])
    return train_idx, val_idx, test_idx

In [ ]:
# ---- Main Training Loop ----

EPOCHS = 200
LR = 1e-3
HIDDEN_DIM = 128
BATCH_SIZE = 64
PATIENCE = 5


def train_and_evaluate(train_g, val_g, test_g, arch, device, seed=42):
    """Train model, return test predictions."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    train_loader = DataLoader(train_g, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_g, batch_size=BATCH_SIZE)
    test_loader = DataLoader(test_g, batch_size=BATCH_SIZE)

    in_dim = train_g[0].x.shape[1]
    if arch == "mlp":
        model = MLPClassifier(in_dim, HIDDEN_DIM).to(device)
    else:
        model = GNNClassifier(in_dim, HIDDEN_DIM, arch=arch).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)

    # Class weights
    n_b = sum(1 for g in train_g if g.y.item() == 0)
    n_a = sum(1 for g in train_g if g.y.item() == 1)
    w = None
    if n_b > 0 and n_a > 0:
        w = torch.tensor([n_a / (n_b + n_a), n_b / (n_b + n_a)], dtype=torch.float).to(device)

    best_val_auroc = 0
    best_state = None
    patience_counter = 0

    for epoch in range(1, EPOCHS + 1):
        train_epoch(model, train_loader, optimizer, device, w)
        if epoch % 10 == 0:
            val_probs, val_labels = evaluate(model, val_loader, device)
            if len(np.unique(val_labels)) > 1:
                val_auroc = roc_auc_score(val_labels, val_probs)
                if val_auroc > best_val_auroc:
                    best_val_auroc = val_auroc
                    patience_counter = 0
                    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                else:
                    patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"    Early stop at epoch {epoch}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    test_probs, test_labels = evaluate(model, test_loader, device)
    return test_probs, test_labels, best_val_auroc

## 7. Run All Experiments

In [ ]:
def run_protocol(sessions, protocol_name, split="label_stratified", mode="content"):
    """Run full evaluation across multiple seeds."""
    sep = "=" * 70
    print(f"\n{sep}")
    print(f"PROTOCOL: {protocol_name} (split={split}, mode={mode}, seeds={SEEDS})")
    print(sep)

    tool_vocab = build_tool_vocab(sessions)
    n_tools = len(tool_vocab)
    print(f"  Tools: {n_tools}")

    embedder = None
    if mode in ("content", "both"):
        print("  Loading sentence transformer...")
        embedder = get_content_embedder()

    print("  Building graphs...")
    graphs = sessions_to_graphs(sessions, mode, tool_vocab, n_tools, embedder)

    labels = np.array([g.y.item() for g in graphs])
    attack_types = [s["attack_type"] for s in sessions]
    sources = [s["source"] for s in sessions]
    task_ids = [s["task_id"] for s in sessions]

    archs = ["gat", "gcn", "sage", "mlp"]
    protocol_results = {}

    for arch in archs:
        print(f"\n  --- {arch.upper()} ---")
        seed_metrics = []

        for seed in SEEDS:
            if split == "task_stratified":
                train_idx, val_idx, test_idx = task_stratified_split(len(graphs), task_ids, seed)
            else:
                train_idx, val_idx, test_idx = label_stratified_split(len(graphs), labels, seed)

            train_g = [graphs[i] for i in train_idx]
            val_g = [graphs[i] for i in val_idx]
            test_g = [graphs[i] for i in test_idx]

            test_probs, test_labels, val_auroc = train_and_evaluate(
                train_g, val_g, test_g, arch, device, seed)

            metrics = compute_metrics(test_probs, test_labels)
            metrics["val_auroc"] = float(val_auroc)
            seed_metrics.append(metrics)

            print(f"    seed={seed}: AUROC={metrics.get('auroc',0):.4f}  "
                  f"MacF1={metrics.get('macro_f1',0):.4f}  "
                  f"Recall={metrics.get('recall',0):.4f}  FPR={metrics.get('fpr',0):.4f}")

        # Aggregate across seeds
        agg = {}
        for m in ["auroc", "auprc", "macro_f1", "weighted_f1", "precision", "recall", "fpr"]:
            vals = [s[m] for s in seed_metrics if m in s]
            if vals:
                agg[f"{m}_mean"] = float(np.mean(vals))
                agg[f"{m}_std"] = float(np.std(vals))
        agg["per_seed"] = seed_metrics

        # Per-attack-type from last seed
        last_seed = SEEDS[-1]
        if split == "task_stratified":
            train_idx, val_idx, test_idx = task_stratified_split(len(graphs), task_ids, last_seed)
        else:
            train_idx, val_idx, test_idx = label_stratified_split(len(graphs), labels, last_seed)
        test_attack_types = [attack_types[i] for i in test_idx]
        test_sources = [sources[i] for i in test_idx]

        train_g = [graphs[i] for i in train_idx]
        val_g = [graphs[i] for i in val_idx]
        test_g = [graphs[i] for i in test_idx]
        test_probs, test_labels, _ = train_and_evaluate(
            train_g, val_g, test_g, arch, device, last_seed)
        agg["per_attack_type"] = per_group_metrics(test_probs, test_labels, test_attack_types)
        agg["per_source"] = per_group_metrics(test_probs, test_labels, test_sources)

        auroc_m = agg.get("auroc_mean", 0)
        auroc_s = agg.get("auroc_std", 0)
        f1_m = agg.get("macro_f1_mean", 0)
        f1_s = agg.get("macro_f1_std", 0)
        rec_m = agg.get("recall_mean", 0)
        rec_s = agg.get("recall_std", 0)
        print(f"\n    MEAN±STD: AUROC={auroc_m:.4f}±{auroc_s:.4f}  "
              f"MacF1={f1_m:.4f}±{f1_s:.4f}  "
              f"Recall={rec_m:.4f}±{rec_s:.4f}")

        protocol_results[arch] = agg

    # Summary
    print(f"\n  {'Arch':<8} {'AUROC':>14} {'AUPRC':>14} {'MacF1':>14} {'Recall':>14} {'FPR':>14}")
    print(f"  {'-'*72}")
    for arch in archs:
        r = protocol_results[arch]
        am = r.get("auroc_mean", 0)
        a_s = r.get("auroc_std", 0)
        pm = r.get("auprc_mean", 0)
        ps = r.get("auprc_std", 0)
        fm = r.get("macro_f1_mean", 0)
        fs = r.get("macro_f1_std", 0)
        rm = r.get("recall_mean", 0)
        rs = r.get("recall_std", 0)
        fpm = r.get("fpr_mean", 0)
        fps = r.get("fpr_std", 0)
        print(f"  {arch.upper():<8} "
              f"{am:.4f}±{a_s:.3f} "
              f"{pm:.4f}±{ps:.3f} "
              f"{fm:.4f}±{fs:.3f} "
              f"{rm:.4f}±{rs:.3f} "
              f"{fpm:.4f}±{fps:.3f}")

    return protocol_results

In [ ]:
# ---- Run all protocols ----
all_results = {}

# Protocol 1: RAS-Eval task-stratified (primary — no task leakage)
ras_sessions = ras_b + ras_a
all_results["ras_eval_task_strat"] = run_protocol(
    ras_sessions, "RAS-Eval", split="task_stratified")

In [ ]:
# Protocol 2: RAS-Eval label-stratified (for comparison)
all_results["ras_eval_label_strat"] = run_protocol(
    ras_sessions, "RAS-Eval", split="label_stratified")

In [ ]:
# Protocol 3: ATBench label-stratified
atb_sessions = atb_b + atb_a
all_results["atbench"] = run_protocol(
    atb_sessions, "ATBench", split="label_stratified")

In [ ]:
# Protocol 4: Combined (RAS-Eval + ATBench + mcpbench-benign)
combined_sessions = ras_b + atb_b + mcp_b + ras_a + atb_a
all_results["combined"] = run_protocol(
    combined_sessions, "Combined (RAS+ATBench+mcpbench)", split="label_stratified")

## 8. Results Summary

In [ ]:
import pandas as pd

rows = []
for protocol, arch_results in all_results.items():
    for arch, metrics in arch_results.items():
        rows.append({
            "Protocol": protocol,
            "Arch": arch.upper(),
            "AUROC": f"{metrics.get('auroc_mean', 0):.4f}±{metrics.get('auroc_std', 0):.3f}",
            "AUPRC": f"{metrics.get('auprc_mean', 0):.4f}±{metrics.get('auprc_std', 0):.3f}",
            "Macro F1": f"{metrics.get('macro_f1_mean', 0):.4f}±{metrics.get('macro_f1_std', 0):.3f}",
            "Precision": f"{metrics.get('precision_mean', 0):.4f}±{metrics.get('precision_std', 0):.3f}",
            "Recall": f"{metrics.get('recall_mean', 0):.4f}±{metrics.get('recall_std', 0):.3f}",
            "FPR": f"{metrics.get('fpr_mean', 0):.4f}±{metrics.get('fpr_std', 0):.3f}",
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))


In [ ]:
# Save results
results_dir = os.path.join(PROJECT_DIR, 'results_colab')
os.makedirs(results_dir, exist_ok=True)
with open(os.path.join(results_dir, 'results.json'), 'w') as f:
  json.dump(all_results, f, indent=2)
  df.to_csv(os.path.join(results_dir, 'summary.csv'), index=False)
  print(f"Results saved to {results_dir}")

## 9. Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: AUROC by protocol and architecture
protocols = list(all_results.keys())
archs = ["gat", "gcn", "sage", "mlp"]
x = np.arange(len(protocols))
width = 0.2

for i, arch in enumerate(archs):
    aurocs = [all_results[p][arch].get("auroc_mean", 0) for p in protocols]
    axes[0].bar(x + i * width, aurocs, width, label=arch.upper())

axes[0].set_xlabel("Protocol")
axes[0].set_ylabel("AUROC")
axes[0].set_title("AUROC by Protocol and Architecture")
axes[0].set_xticks(x + 1.5 * width)
axes[0].set_xticklabels([p.replace('_', '\n') for p in protocols], fontsize=8)
axes[0].legend()
axes[0].set_ylim(0.5, 1.0)
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: Macro F1 comparison
for i, arch in enumerate(archs):
    f1s = [all_results[p][arch].get("macro_f1_mean", 0) for p in protocols]
    axes[1].bar(x + i * width, f1s, width, label=arch.upper())

axes[1].set_xlabel("Protocol")
axes[1].set_ylabel("Macro F1")
axes[1].set_title("Macro F1 by Protocol and Architecture")
axes[1].set_xticks(x + 1.5 * width)
axes[1].set_xticklabels([p.replace('_', '\n') for p in protocols], fontsize=8)
axes[1].legend()
axes[1].set_ylim(0.5, 1.0)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-attack-type heatmap for best architecture on combined
best_arch = max(all_results["combined"].keys(),
                key=lambda a: all_results["combined"][a].get("auroc", 0))
atk_data = all_results["combined"][best_arch]["per_attack_type"]

print(f"\nBest architecture on Combined: {best_arch.upper()}")
print(f"\n{'Attack Type':<45} {'Recall':>8} {'N':>6}")
print("-" * 62)
for name, stats in sorted(atk_data.items(), key=lambda x: x[1].get('recall', -1), reverse=True):
    recall = stats.get('recall', 'N/A')
    if isinstance(recall, float):
        recall = f"{recall:.3f}"
    print(f"{name:<45} {recall:>8} {stats['n_total']:>6}")

## 10. Label Efficiency Experiment

SSL+fine-tune vs supervised-from-scratch at varying label fractions (1%, 5%, 10%, 25%, 50%, 100%).
Run on **all three datasets**: RAS-Eval (task-stratified), ATBench (label-stratified), Combined (label-stratified).

In [ ]:
# ---- SSL Components ----
from torch_geometric.nn import GATConv as GATConv2

class GATEncoder(torch.nn.Module):
    """GAT encoder for SSL pre-training."""
    def __init__(self, in_dim, hidden_dim, heads=4):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden_dim, heads=heads, concat=False)
        self.conv2 = GATConv(hidden_dim, hidden_dim, heads=heads, concat=False)

    def forward(self, x, edge_index, batch):
        x = F.elu(self.conv1(x, edge_index))
        x = F.elu(self.conv2(x, edge_index))
        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)
        return torch.cat([x_mean, x_max], dim=1)


class ProjectionHead(torch.nn.Module):
    def __init__(self, in_dim, out_dim=64):
        super().__init__()
        self.lin1 = torch.nn.Linear(in_dim, in_dim)
        self.lin2 = torch.nn.Linear(in_dim, out_dim)

    def forward(self, x):
        return self.lin2(F.relu(self.lin1(x)))


class ClassificationHead(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.lin1 = torch.nn.Linear(in_dim, hidden_dim)
        self.lin2 = torch.nn.Linear(hidden_dim, 2)
        self.dropout = torch.nn.Dropout(0.3)

    def forward(self, x):
        x = F.elu(self.lin1(x))
        x = self.dropout(x)
        return self.lin2(x)


def augment_graph(data, mask_rate=0.2, drop_edge_rate=0.2):
    """Augment graph for contrastive learning."""
    import copy
    aug = data.clone()
    if mask_rate > 0:
        mask = torch.rand(aug.x.shape) > mask_rate
        aug.x = aug.x * mask.float().to(aug.x.device)
    if drop_edge_rate > 0 and aug.edge_index.shape[1] > 1:
        n_edges = aug.edge_index.shape[1]
        keep = torch.rand(n_edges) > drop_edge_rate
        if keep.sum() > 0:
            aug.edge_index = aug.edge_index[:, keep.to(aug.edge_index.device)]
    return aug


def nt_xent_loss(z1, z2, temperature=0.5):
    """NT-Xent contrastive loss."""
    z = torch.cat([z1, z2], dim=0)
    z = F.normalize(z, dim=1)
    n = z1.shape[0]
    sim = torch.mm(z, z.t()) / temperature
    mask = torch.eye(2 * n, dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(mask, -1e9)
    pos = torch.cat([torch.diag(sim, n), torch.diag(sim, -n)])
    return -pos.mean() + torch.logsumexp(sim, dim=1).mean()


print("SSL components loaded.")

In [ ]:
# ---- Label Efficiency Helper Functions ----

LABEL_FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]


def subsample_labeled(graphs, fraction, rng):
    """Subsample a fraction of labeled graphs, preserving class ratio."""
    if fraction >= 1.0:
        return graphs
    benign = [g for g in graphs if g.y.item() == 0]
    attack = [g for g in graphs if g.y.item() == 1]
    n_b = max(1, int(len(benign) * fraction))
    n_a = max(1, int(len(attack) * fraction))
    idx_b = rng.permutation(len(benign))[:n_b]
    idx_a = rng.permutation(len(attack))[:n_a]
    return [benign[i] for i in idx_b] + [attack[i] for i in idx_a]


def pretrain_ssl(encoder, proj_head, benign_graphs, pretrain_epochs, batch_size,
                 lr, mask_rate=0.2, drop_edge_rate=0.2, temperature=0.5):
    """Pre-train encoder on benign graphs with contrastive learning."""
    loader = DataLoader(benign_graphs, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(
        list(encoder.parameters()) + list(proj_head.parameters()),
        lr=lr, weight_decay=1e-4)

    for epoch in range(1, pretrain_epochs + 1):
        encoder.train()
        proj_head.train()
        for batch in loader:
            batch = batch.to(device)
            aug1 = augment_graph(batch, mask_rate, drop_edge_rate)
            aug2 = augment_graph(batch, mask_rate, drop_edge_rate)
            h1 = encoder(aug1.x, aug1.edge_index, aug1.batch)
            h2 = encoder(aug2.x, aug2.edge_index, aug2.batch)
            z1 = proj_head(h1)
            z2 = proj_head(h2)
            loss = nt_xent_loss(z1, z2, temperature)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(encoder.parameters()) + list(proj_head.parameters()), 1.0)
            optimizer.step()


def train_classifier_le(model_or_encoder, cls_head, train_graphs, val_graphs,
                        max_epochs, lr, batch_size, is_ssl=False, freeze_encoder=False):
    """Train classifier for label efficiency experiment."""
    train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_graphs, batch_size=batch_size)

    n_b = sum(1 for g in train_graphs if g.y.item() == 0)
    n_a = sum(1 for g in train_graphs if g.y.item() == 1)
    w = None
    if n_b > 0 and n_a > 0:
        w = torch.tensor([n_a / (n_b + n_a), n_b / (n_b + n_a)], dtype=torch.float).to(device)

    if is_ssl:
        params = list(cls_head.parameters()) if freeze_encoder else \
                 list(model_or_encoder.parameters()) + list(cls_head.parameters())
    else:
        params = list(model_or_encoder.parameters())
    optimizer = torch.optim.Adam(params, lr=lr, weight_decay=1e-4)

    best_val_auroc = 0
    best_state = None
    patience_counter = 0

    for epoch in range(1, max_epochs + 1):
        if is_ssl:
            if freeze_encoder:
                model_or_encoder.eval()
            else:
                model_or_encoder.train()
            cls_head.train()
        else:
            model_or_encoder.train()

        for batch in train_loader:
            batch = batch.to(device)
            if is_ssl:
                if freeze_encoder:
                    with torch.no_grad():
                        h = model_or_encoder(batch.x, batch.edge_index, batch.batch)
                else:
                    h = model_or_encoder(batch.x, batch.edge_index, batch.batch)
                out = cls_head(h)
            else:
                out = model_or_encoder(batch)
            loss = F.cross_entropy(out, batch.y.view(-1), weight=w)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step()

        if epoch % 5 == 0:
            model_or_encoder.eval()
            if is_ssl:
                cls_head.eval()
            all_probs, all_labels = [], []
            with torch.no_grad():
                for batch in val_loader:
                    batch = batch.to(device)
                    if is_ssl:
                        h = model_or_encoder(batch.x, batch.edge_index, batch.batch)
                        out = cls_head(h)
                    else:
                        out = model_or_encoder(batch)
                    probs = F.softmax(out, dim=1)[:, 1]
                    all_probs.extend(probs.cpu().numpy())
                    all_labels.extend(batch.y.view(-1).cpu().numpy())
            if len(np.unique(all_labels)) > 1:
                val_auroc = roc_auc_score(all_labels, np.array(all_probs))
                if val_auroc > best_val_auroc:
                    best_val_auroc = val_auroc
                    patience_counter = 0
                    if is_ssl:
                        best_state = {
                            "encoder": {k: v.cpu().clone() for k, v in model_or_encoder.state_dict().items()},
                            "cls_head": {k: v.cpu().clone() for k, v in cls_head.state_dict().items()},
                        }
                    else:
                        best_state = {k: v.cpu().clone() for k, v in model_or_encoder.state_dict().items()}
                else:
                    patience_counter += 1
            if patience_counter >= 10:
                break

    if best_state is not None:
        if is_ssl:
            model_or_encoder.load_state_dict(best_state["encoder"])
            cls_head.load_state_dict(best_state["cls_head"])
        else:
            model_or_encoder.load_state_dict(best_state)
    return best_val_auroc


def evaluate_le(model_or_encoder, cls_head, test_graphs, is_ssl=False):
    """Evaluate model, return AUROC, F1, Recall, FPR."""
    loader = DataLoader(test_graphs, batch_size=64)
    model_or_encoder.eval()
    if is_ssl:
        cls_head.eval()

    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            if is_ssl:
                h = model_or_encoder(batch.x, batch.edge_index, batch.batch)
                out = cls_head(h)
            else:
                out = model_or_encoder(batch)
            probs = F.softmax(out, dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(batch.y.view(-1).cpu().numpy())

    probs = np.array(all_probs)
    labels = np.array(all_labels)
    results = {}
    if len(np.unique(labels)) > 1:
        results["auroc"] = float(roc_auc_score(labels, probs))
        preds = (probs > 0.5).astype(int)
        results["f1"] = float(f1_score(labels, preds, average="macro"))
        results["recall"] = float(recall_score(labels, preds))
        tp = ((preds == 1) & (labels == 1)).sum()
        fp = ((preds == 1) & (labels == 0)).sum()
        tn = ((preds == 0) & (labels == 0)).sum()
        results["fpr"] = float(fp / max(fp + tn, 1))
    return results


print("Label efficiency helpers loaded.")

In [ ]:
# ---- Run Label Efficiency on All Datasets ----
import copy
from sklearn.model_selection import GroupKFold, StratifiedKFold

PRETRAIN_EPOCHS = 100
FINETUNE_EPOCHS = 50
SUPERVISED_EPOCHS = 100
LE_LR = 1e-3
LE_HIDDEN = 128
LE_BATCH = 64
LE_SEED = 42


def run_label_efficiency_fold(train_graphs, test_graphs, all_benign, in_dim,
                              fold_idx, fraction):
    """Run one fold: SSL (linear probe + fine-tune) vs supervised."""
    rng = np.random.RandomState(LE_SEED + fold_idx + int(fraction * 1000))

    indices = rng.permutation(len(train_graphs))
    n_val = max(1, len(indices) // 5)
    val_graphs = [train_graphs[i] for i in indices[:n_val]]
    trn_full = [train_graphs[i] for i in indices[n_val:]]

    trn_labeled = subsample_labeled(trn_full, fraction, rng)
    n_b = sum(1 for g in trn_labeled if g.y.item() == 0)
    n_a = sum(1 for g in trn_labeled if g.y.item() == 1)

    # SSL: pre-train encoder on ALL benign (no labels needed)
    encoder_ssl = GATEncoder(in_dim, LE_HIDDEN).to(device)
    proj_head = ProjectionHead(LE_HIDDEN * 2, 64).to(device)
    pretrain_ssl(encoder_ssl, proj_head, all_benign, PRETRAIN_EPOCHS, LE_BATCH, LE_LR)

    # SSL + linear probe (freeze encoder)
    encoder_frozen = copy.deepcopy(encoder_ssl)
    cls_lp = ClassificationHead(LE_HIDDEN * 2, LE_HIDDEN).to(device)
    train_classifier_le(encoder_frozen, cls_lp, trn_labeled, val_graphs,
                        FINETUNE_EPOCHS, LE_LR, LE_BATCH, is_ssl=True, freeze_encoder=True)
    lp_results = evaluate_le(encoder_frozen, cls_lp, test_graphs, is_ssl=True)

    # SSL + full fine-tune
    cls_ft = ClassificationHead(LE_HIDDEN * 2, LE_HIDDEN).to(device)
    train_classifier_le(encoder_ssl, cls_ft, trn_labeled, val_graphs,
                        FINETUNE_EPOCHS, LE_LR * 0.1, LE_BATCH, is_ssl=True, freeze_encoder=False)
    ssl_results = evaluate_le(encoder_ssl, cls_ft, test_graphs, is_ssl=True)

    # Supervised from scratch
    model_sup = GNNClassifier(in_dim, LE_HIDDEN, arch='gat').to(device)
    train_classifier_le(model_sup, None, trn_labeled, val_graphs,
                        SUPERVISED_EPOCHS, LE_LR, LE_BATCH, is_ssl=False)
    sup_results = evaluate_le(model_sup, None, test_graphs, is_ssl=False)

    print(f'    Fold {fold_idx} ({fraction*100:.0f}%): '
          f'LP={lp_results.get("auroc",0):.4f}  '
          f'FT={ssl_results.get("auroc",0):.4f}  '
          f'Sup={sup_results.get("auroc",0):.4f}  '
          f'(labeled: {n_b}b+{n_a}a)')

    return {'linear_probe': lp_results, 'ssl_finetune': ssl_results,
            'supervised': sup_results, 'n_labeled': n_b + n_a}


def run_label_efficiency(sessions, dataset_name, split_mode='label_stratified'):
    """Run full label efficiency experiment on one dataset."""
    sep = '=' * 70
    print(f'\n{sep}')
    print(f'LABEL EFFICIENCY: {dataset_name} (split={split_mode})')
    print(sep)

    tool_vocab = build_tool_vocab(sessions)
    n_tools = len(tool_vocab)
    print(f'  Tools: {n_tools}, Sessions: {len(sessions)}')

    print('  Loading embedder...')
    embedder = get_content_embedder()
    print('  Building graphs...')
    graphs = sessions_to_graphs(sessions, 'content', tool_vocab, n_tools, embedder)
    in_dim = graphs[0].x.shape[1]

    labels = np.array([g.y.item() for g in graphs])
    task_ids = [s['task_id'] for s in sessions]

    np.random.seed(LE_SEED)
    torch.manual_seed(LE_SEED)

    if split_mode == 'task_stratified':
        kf = GroupKFold(n_splits=5)
        splits = list(kf.split(graphs, labels, task_ids))
    else:
        kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=LE_SEED)
        splits = list(kf.split(graphs, labels))

    dataset_results = {}
    for fraction in LABEL_FRACTIONS:
        print(f'\n  --- {fraction*100:.0f}% labels ---')
        fold_results = []
        for fold_idx, (train_idx, test_idx) in enumerate(splits):
            train_g = [graphs[i] for i in train_idx]
            test_g = [graphs[i] for i in test_idx]
            all_benign = [g for g in train_g if g.y.item() == 0]

            r = run_label_efficiency_fold(train_g, test_g, all_benign, in_dim,
                                          fold_idx, fraction)
            fold_results.append(r)

        for method in ['linear_probe', 'ssl_finetune', 'supervised']:
            aurocs = [r[method].get('auroc', 0) for r in fold_results]
            print(f'    {method}: AUROC={np.mean(aurocs):.4f}\u00b1{np.std(aurocs):.4f}')

        dataset_results[f'{fraction*100:.0f}%'] = {
            'folds': fold_results,
            'lp_auroc_mean': float(np.mean([r['linear_probe'].get('auroc', 0) for r in fold_results])),
            'lp_auroc_std': float(np.std([r['linear_probe'].get('auroc', 0) for r in fold_results])),
            'ft_auroc_mean': float(np.mean([r['ssl_finetune'].get('auroc', 0) for r in fold_results])),
            'ft_auroc_std': float(np.std([r['ssl_finetune'].get('auroc', 0) for r in fold_results])),
            'sup_auroc_mean': float(np.mean([r['supervised'].get('auroc', 0) for r in fold_results])),
            'sup_auroc_std': float(np.std([r['supervised'].get('auroc', 0) for r in fold_results])),
        }

    return dataset_results


# ---- Run on all three datasets ----
le_results = {}

# 1. RAS-Eval (task-stratified — primary, no task leakage)
le_results['ras_eval'] = run_label_efficiency(
    ras_b + ras_a, 'RAS-Eval', split_mode='task_stratified')

# 2. ATBench (label-stratified)
le_results['atbench'] = run_label_efficiency(
    atb_b + atb_a, 'ATBench', split_mode='label_stratified')

# 3. Combined (label-stratified)
le_results['combined'] = run_label_efficiency(
    ras_b + atb_b + mcp_b + ras_a + atb_a, 'Combined', split_mode='label_stratified')


In [ ]:
# ---- Label Efficiency Summary & Visualization ----

print('\n' + '=' * 80)
print('LABEL EFFICIENCY SUMMARY')
print('=' * 80)

for dataset_name, ds_results in le_results.items():
    print(f'\n--- {dataset_name} ---')
    print(f'{"Labels":>8}  {"LinProbe":>14}  {"SSL+FT":>14}  {"Supervised":>14}  {"FT-Sup":>8}')
    print('-' * 70)
    for frac_label, r in ds_results.items():
        lp_m = r['lp_auroc_mean']
        ft_m = r['ft_auroc_mean']
        sup_m = r['sup_auroc_mean']
        delta = ft_m - sup_m
        sign = '+' if delta > 0 else ''
        print(f'{frac_label:>8}  '
              f'{lp_m:.4f}\u00b1{r["lp_auroc_std"]:.3f}  '
              f'{ft_m:.4f}\u00b1{r["ft_auroc_std"]:.3f}  '
              f'{sup_m:.4f}\u00b1{r["sup_auroc_std"]:.3f}  '
              f'{sign}{delta:.4f}')

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fractions_pct = [1, 5, 10, 25, 50, 100]

for ax_idx, (dataset_name, ds_results) in enumerate(le_results.items()):
    ax = axes[ax_idx]
    lp_means = [ds_results[f'{f}%']['lp_auroc_mean'] for f in fractions_pct]
    ft_means = [ds_results[f'{f}%']['ft_auroc_mean'] for f in fractions_pct]
    sup_means = [ds_results[f'{f}%']['sup_auroc_mean'] for f in fractions_pct]
    lp_stds = [ds_results[f'{f}%']['lp_auroc_std'] for f in fractions_pct]
    ft_stds = [ds_results[f'{f}%']['ft_auroc_std'] for f in fractions_pct]
    sup_stds = [ds_results[f'{f}%']['sup_auroc_std'] for f in fractions_pct]

    x = np.arange(len(fractions_pct))
    ax.errorbar(x, lp_means, yerr=lp_stds, marker='s', label='SSL Linear Probe', capsize=3)
    ax.errorbar(x, ft_means, yerr=ft_stds, marker='^', label='SSL + Fine-tune', capsize=3)
    ax.errorbar(x, sup_means, yerr=sup_stds, marker='o', label='Supervised', capsize=3)

    ax.set_xticks(x)
    ax.set_xticklabels([f'{f}%' for f in fractions_pct])
    ax.set_xlabel('Label Fraction')
    ax.set_ylabel('AUROC')
    ax.set_title(f'{dataset_name}')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_ylim(0.4, 1.0)

plt.suptitle('Label Efficiency: SSL vs Supervised', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'label_efficiency.png'), dpi=150, bbox_inches='tight')
plt.show()

# Save results
with open(os.path.join(results_dir, 'label_efficiency_results.json'), 'w') as f:
    json.dump(le_results, f, indent=2, default=str)
print(f'\nLabel efficiency results saved to {results_dir}/label_efficiency_results.json')
